In [ ]:
import pyscf
from pyscf import gto, dft, tdscf
import numpy as np  # For coordinate conversion

# Lattice parameters from Materials Project (mp-16969)
a = 14.17  # Å
b = 6.59
c = 10.13
alpha = 90.0 * np.pi / 180
beta = 122.08 * np.pi / 180
gamma = 90.0 * np.pi / 180

# Conversion matrix from fractional to Cartesian for monoclinic (alpha=gamma=90, beta!=90)
# Lattice vectors: A = (a, 0, 0), B = (0, b, 0), C = (c*cos(beta), 0, c*sin(beta))
cos_beta = np.cos(beta)
sin_beta = np.sin(beta)
to_cart = np.array([
    [a, 0, c * cos_beta],
    [0, b, 0],
    [0, 0, c * sin_beta]
])

# Fractional coordinates from Materials Project (unique Wyckoff sites; approximate one formula unit by selecting nearby atoms)
# For simplicity: Ce (replacing Lu1 at site 1), Lu (site 2), Si (site 1), and 5 O's (sites 1-5)
# Real cluster would optimize and embed; these are direct conversions
frac_coords = np.array([
    [0.85969, 0.622987, 0.66254],  # Ce (was Lu1)
    [0.962792, 0.743551, 0.031843],  # Lu (Lu2)
    [0.682614, 0.407105, 0.307063],  # Si
    [0.982503, 0.59732, 0.602532],   # O1
    [0.202412, 0.072826, 0.43799],   # O2
    [0.799054, 0.34951, 0.322791],   # O3
    [0.088013, 0.008496, 0.635969],  # O4
    [0.120335, 0.291022, 0.82618]    # O5
])

# Convert to Cartesian (Å)
cart_coords = np.dot(frac_coords, to_cart)

# Build atom string for PySCF
atom_list = [
    f'Ce {cart_coords[0][0]} {cart_coords[0][1]} {cart_coords[0][2]}',
    f'Lu {cart_coords[1][0]} {cart_coords[1][1]} {cart_coords[1][2]}',
    f'Si {cart_coords[2][0]} {cart_coords[2][1]} {cart_coords[2][2]}',
    f'O  {cart_coords[3][0]} {cart_coords[3][1]} {cart_coords[3][2]}',
    f'O  {cart_coords[4][0]} {cart_coords[4][1]} {cart_coords[4][2]}',
    f'O  {cart_coords[5][0]} {cart_coords[5][1]} {cart_coords[5][2]}',
    f'O  {cart_coords[6][0]} {cart_coords[6][1]} {cart_coords[6][2]}',
    f'O  {cart_coords[7][0]} {cart_coords[7][1]} {cart_coords[7][2]}'
]
atom_str = '; '.join(atom_list)

# Define molecule (cluster)
mol = gto.M(
    atom=atom_str,
    basis='def2-svp',  # Split-valence basis; use 'def2-tzvp' for better accuracy
    ecp={'Ce': 'stuttgart_rsc_1997_ecp', 'Lu': 'stuttgart_rsc_1997_ecp'},  # ECP for heavy atoms (PySCF built-in)
    spin=1,  # For Ce3+ (4f1, odd electron)
    charge=0,  # Neutral cluster; adjust if needed for ionic model
    verbose=4  # For debug output
)

# Ground-state DFT (use PBE; for better gaps, use 'hse06' hybrid)
mf = dft.RKS(mol)
mf.xc = 'pbe'  # Or 'hse06' for screened hybrid (better for excitations)
mf.kernel()  # Run SCF

# TDDFT for excited states
td = tdscf.TDDFT(mf)
td.nstates = 10  # Compute 10 lowest excitations (adjust for Ce 4f->5d ~3-5 eV)
td.kernel()

# Print results (excitation energies in eV, oscillator strengths)
print('Excitation energies (eV):', td.e * 27.2114)  # Hartree to eV
print('Oscillator strengths:', td.oscillator_strength())

# Optional: Analyze transitions (e.g., for emission, use singlet/triplet)
# td.analyze()